# Aqua / MODIS (Moderate Resolution Imaging Spectroradiometer)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=modis/modis-aqua). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1B

Level 1B: calibrated and geolocated radiances at 250 m, 500 m and 1 km (MYD021KM, MYD02HKM, MYD02QKM) plus the geolocation file MYD03.

Searches NASA Earthdata for a MYD021KM granule over central Italy, downloads it (needs a free Earthdata login, asked at run time) and plots band 1 (red) reflectance. HDF4 files are read with pyhdf.

In [ ]:
%pip install -q earthaccess pyhdf matplotlib numpy

In [ ]:
from pathlib import Path

import earthaccess
import matplotlib.pyplot as plt
import numpy as np
from pyhdf.SD import SD, SDC

earthaccess.login()   # asks for a (free) NASA Earthdata Login the first time

results = earthaccess.search_data(
    short_name="MYD021KM",
    version="61",
    bounding_box=(12.0, 41.0, 13.0, 42.5),       # west, south, east, north (central Italy)
    temporal=("2024-06-01", "2024-06-02"),
)
print(len(results), "granules found")
files = earthaccess.download(results[:1], "data")
path = Path(files[0])

hdf = SD(str(path), SDC.READ)
print(sorted(hdf.datasets()))

sds = hdf.select("EV_250_Aggr1km_RefSB")
attrs = sds.attributes()
dn = sds[0, :, :].astype("float32")
refl = (dn - attrs["reflectance_offsets"][0]) * attrs["reflectance_scales"][0]
refl[dn > 32767] = np.nan                         # fill / invalid values
plt.imshow(refl, cmap="gray", vmin=0, vmax=np.nanpercentile(refl, 98))
plt.colorbar(label="reflectance")
plt.title(path.name + " - band 1 reflectance")
plt.show()

## L2

Level 2: geophysical swath products (granule level): aerosol (MYD04), water vapour (MYD05), cloud (MYD06), atmospheric profiles (MYD07), cloud mask (MYD35), LST (MYD11_L2), fires (MYD14), sea surface temperature and ocean colour.

Searches NASA Earthdata for a MYD11_L2 (land surface temperature) granule over central Italy, downloads it (needs a free Earthdata login, asked at run time) and plots the LST swath. HDF4 files are read with pyhdf.

In [ ]:
%pip install -q earthaccess pyhdf matplotlib numpy

In [ ]:
from pathlib import Path

import earthaccess
import matplotlib.pyplot as plt
import numpy as np
from pyhdf.SD import SD, SDC

earthaccess.login()   # asks for a (free) NASA Earthdata Login the first time

results = earthaccess.search_data(
    short_name="MYD11_L2",
    version="061",
    bounding_box=(12.0, 41.0, 13.0, 42.5),       # west, south, east, north (central Italy)
    temporal=("2024-06-01", "2024-06-02"),
)
print(len(results), "granules found")
files = earthaccess.download(results[:1], "data")
path = Path(files[0])

hdf = SD(str(path), SDC.READ)
print(sorted(hdf.datasets()))

sds = hdf.select("LST")
raw = sds[:, :].astype("float32")
raw[raw == 0] = np.nan                           # fill / invalid values
data = raw * 0.02
plt.imshow(data, cmap="inferno")
plt.colorbar(label="land surface temperature (K)")
plt.title(path.name)
plt.show()

## L3

Level 3: gridded daily, 8-day, 16-day and monthly land products on the sinusoidal grid: surface reflectance (MYD09), LST (MYD11), vegetation indices (MYD13), snow (MYD10), thermal anomalies (MYD14).

Searches the Microsoft Planetary Computer STAC catalogue (no account needed) for the daily MYD11A1 land surface temperature tile over Rome and plots daytime LST in kelvin.

In [ ]:
%pip install -q pystac-client planetary-computer rioxarray matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import planetary_computer
import pystac_client
import rioxarray

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)


def open_overview(href, level=2):
    """Open a COG at reduced resolution (fast), falling back to full resolution."""
    try:
        return rioxarray.open_rasterio(href, overview_level=level, masked=True).squeeze()
    except Exception:
        return rioxarray.open_rasterio(href, masked=True).squeeze()

search = catalog.search(
    collections=["modis-11A1-061"],
    bbox=[12.4, 41.8, 12.6, 42.0],                      # Rome - change to your area
    datetime="2024-07-01/2024-07-10",
)
items = [i for i in search.item_collection() if i.id.startswith("MYD11A1")]
if not items:
    raise SystemExit("No tiles found - widen the dates or the bounding box")
item = items[0]
print(len(items), "tiles;", "using", item.id, item.datetime)
print("assets:", sorted(item.assets))

key = next((k for k in item.assets if "LST_Day" in k), None)
if key is None:
    raise SystemExit("No asset containing 'LST_Day' - pick one from the list above")
data = open_overview(item.assets[key].href, level=1) * 0.02
data.plot(cmap="inferno" if "K" == "K" else "YlGn")
plt.title(item.id + " - " + key + " (land surface temperature (K))")
plt.show()

## L4

Level 4: modelled and derived land products: LAI/FPAR (MYD15A2H), evapotranspiration (MYD16A2), gross and net primary production (MYD17A2H, MYD17A3HGF), land cover dynamics.

Searches the Microsoft Planetary Computer STAC catalogue (no account needed) for the 8-day MYD17A2H gross primary production tile over Rome and plots GPP.

In [ ]:
%pip install -q pystac-client planetary-computer rioxarray matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import planetary_computer
import pystac_client
import rioxarray

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)


def open_overview(href, level=2):
    """Open a COG at reduced resolution (fast), falling back to full resolution."""
    try:
        return rioxarray.open_rasterio(href, overview_level=level, masked=True).squeeze()
    except Exception:
        return rioxarray.open_rasterio(href, masked=True).squeeze()

search = catalog.search(
    collections=["modis-17A2H-061"],
    bbox=[12.4, 41.8, 12.6, 42.0],                      # Rome - change to your area
    datetime="2024-07-01/2024-07-20",
)
items = [i for i in search.item_collection() if i.id.startswith("MYD17A2H")]
if not items:
    raise SystemExit("No tiles found - widen the dates or the bounding box")
item = items[0]
print(len(items), "tiles;", "using", item.id, item.datetime)
print("assets:", sorted(item.assets))

key = next((k for k in item.assets if "Gpp" in k), None)
if key is None:
    raise SystemExit("No asset containing 'Gpp' - pick one from the list above")
data = open_overview(item.assets[key].href, level=1) * 0.0001
data.plot(cmap="inferno" if "V" == "K" else "YlGn")
plt.title(item.id + " - " + key + " (GPP (kg C/m2/8 days))")
plt.show()